# SmartFruit — Baseline de visión

Notebook reorganizado a partir del código original compartido.

## Criterio de organización

Baseline de visión: auditoría de etiquetas y particiones, inspección visual, experimentos YOLO, trabajo con ResNet18 cuando aparece en el original, construcción reproducible del dataset V3, entrenamiento, recuperación de artefactos y evaluación final en test.

**Se conserva el código y los apartados del material original.** La organización separa secciones para facilitar la lectura; revisa las rutas y dependencias antes de ejecutar.

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1vWACPi0aYTR95dsw-3s80dRCeLE2yWwQ

## **BASELINE VERSIÓN 1 (YOLO)**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

base = "/content/drive/MyDrive"

for carpeta, subcarpetas, archivos in os.walk(base):
    for archivo in archivos:
        if archivo.lower().endswith(".zip"):
            print(os.path.join(carpeta, archivo))

import os
import zipfile

zip_path = "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/PROYECTO INTEGRADOR/SMARTFRUIT MODELO/dataset_oleocelosis_yolo.zip"

extract_dir = "/content/smartfruit_dataset"

if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"No se encontró el ZIP en la ruta indicada:\n{zip_path}"
    )

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Dataset extraído en:", extract_dir)

for carpeta, subcarpetas, archivos in os.walk(extract_dir):
    nivel = carpeta.replace(extract_dir, "").count(os.sep)
    if nivel <= 3:
        indent = "  " * nivel
        print(f"{indent}{os.path.basename(carpeta)}/")
        for archivo in archivos[:5]:
            print(f"{indent}  {archivo}")
        if len(archivos) > 5:
            print(f"{indent}  ... y {len(archivos)-5} archivos más")

from pathlib import Path

root = Path("/content/smartfruit_dataset")

imagenes = list(root.rglob("*.jpg")) + list(root.rglob("*.JPG"))
etiquetas = list(root.rglob("*.txt")) + list(root.rglob("*.TXT"))

print("Total de imágenes JPG:", len(imagenes))
print("Total de archivos TXT:", len(etiquetas))

for split in ["train", "val", "test"]:
    imgs = [
        p for p in imagenes
        if split in p.parts
    ]
    labs = [
        p for p in etiquetas
        if split in p.parts
    ]
    print(f"{split}: {len(imgs)} imágenes | {len(labs)} etiquetas")

from pathlib import Path
from collections import Counter, defaultdict
import yaml

root = Path("/content/smartfruit_dataset/dataset_oleocelosis_yolo")

## 1. Revisar la configuración de YOLO

In [ ]:
yaml_path = root / "data.yaml"

print("=== CONFIGURACIÓN data.yaml ===")
print(yaml_path.read_text())

## 2. Contar las clases anotadas

In [ ]:
for split in ["train", "val"]:
    label_dir = root / "labels" / split
    conteo = Counter()
    vacias = 0
    mal_formadas = []

    for txt in label_dir.glob("*.txt"):
        lineas = [
            linea.strip()
            for linea in txt.read_text().splitlines()
            if linea.strip()
        ]

        if not lineas:
            vacias += 1
            continue

        for linea in lineas:
            partes = linea.split()
            if len(partes) != 5:
                mal_formadas.append(txt.name)
                continue
            try:
                clase = int(partes[0])
                conteo[clase] += 1
            except ValueError:
                mal_formadas.append(txt.name)

    print(f"\n=== {split.upper()} ===")
    print("Anotaciones por clase:", dict(sorted(conteo.items())))
    print("Etiquetas vacías:", vacias)
    print("Archivos con formato inesperado:", len(set(mal_formadas)))

## 3. Identificar fotografías originales por el patrón IMG_XXXX_LNN

In [ ]:
def foto_original(nombre):
    return nombre.split("_L")[0]

grupos = {}

for split in ["train", "val"]:
    image_dir = root / "images" / split
    grupos[split] = {
        foto_original(p.stem) for p in image_dir.glob("*.jpg")
    }

cruce = grupos["train"] & grupos["val"]

print("\n=== SEPARACIÓN POR FOTOGRAFÍA ORIGINAL ===")
print("Fotos originales representadas en train:", len(grupos["train"]))
print("Fotos originales representadas en val:", len(grupos["val"]))
print("Fotos originales presentes en ambos:", len(cruce))
print("Ejemplos de cruce:", sorted(cruce)[:15])

from pathlib import Path
from collections import Counter

root = Path("/content/smartfruit_dataset/dataset_oleocelosis_yolo")

for split in ["train", "val"]:
    label_dir = root / "labels" / split
    image_dir = root / "images" / split

    cantidad_cajas = Counter()
    imagenes_por_clase = Counter()
    archivos_multi = []
    archivos_mixtos = []
    faltan_txt = []
    faltan_img = []

    for img in image_dir.glob("*"):
        if img.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue
        txt = label_dir / f"{img.stem}.txt"
        if not txt.exists():
            faltan_txt.append(img.name)

    for txt in label_dir.glob("*.txt"):
        img_exists = any(
            (image_dir / f"{txt.stem}{ext}").exists()
            for ext in [".jpg", ".JPG", ".jpeg", ".JPEG", ".png", ".PNG"]
        )
        if not img_exists:
            faltan_img.append(txt.name)

        lineas = [l.split() for l in txt.read_text().splitlines() if l.strip()]
        clases = [int(l[0]) for l in lineas if len(l) == 5]

        cantidad_cajas[len(clases)] += 1
        for clase in set(clases):
            imagenes_por_clase[clase] += 1

        if len(clases) > 1:
            archivos_multi.append((txt.name, clases))
        if len(set(clases)) > 1:
            archivos_mixtos.append((txt.name, clases))

    print(f"\n--- {split.upper()} ---")
    print("Imágenes con 1 caja, 2 cajas, etc.:", dict(sorted(cantidad_cajas.items())))
    print("Imágenes con al menos una etiqueta por clase:", dict(imagenes_por_clase))
    print("Imágenes con más de una caja:", len(archivos_multi))
    print("Imágenes con ambas clases:", len(archivos_mixtos))
    print("Imágenes sin TXT:", len(faltan_txt))
    print("TXT sin imagen:", len(faltan_img))
    print("Ejemplos con varias cajas:", archivos_multi[:5])

from pathlib import Path
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

root = Path("/content/smartfruit_dataset/dataset_oleocelosis_yolo")
image_dir = root / "images" / "train"
label_dir = root / "labels" / "train"

# Priorizar ejemplos con varias cajas
candidatas = []
for txt in label_dir.glob("*.txt"):
    lineas = [l.split() for l in txt.read_text().splitlines() if l.strip()]
    if len(lineas) >= 2:
        img = image_dir / f"{txt.stem}.jpg"
        if img.exists():
            candidatas.append((img, txt))

random.seed(42)
muestra = random.sample(candidatas, min(6, len(candidatas)))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, (img_path, txt_path) in zip(axes, muestra):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    ax.imshow(img)

    lineas = [l.split() for l in txt_path.read_text().splitlines() if l.strip()]
    for partes in lineas:
        clase = int(partes[0])
        xc, yc, bw, bh = map(float, partes[1:5])

        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        rect = patches.Rectangle(
            (x1, y1), bw * w, bh * h,
            linewidth=2, edgecolor="red", facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            x1, y1, f"Clase {clase}",
            color="white", fontsize=9,
            bbox=dict(facecolor="red", alpha=0.7)
        )

    ax.set_title(img_path.name)
    ax.axis("off")

for ax in axes[len(muestra):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

from pathlib import Path

yaml_path = Path(
    "/content/smartfruit_dataset/dataset_oleocelosis_yolo/data.yaml"
)

print(yaml_path.read_text())

## 1. Instalar Ultralytics YOLO

In [ ]:
!pip install -q ultralytics

## 2. Comprobar GPU

In [ ]:
import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No hay GPU activa. En Colab, ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU.")

from pathlib import Path
import yaml

dataset_dir = Path(
    "/content/smartfruit_dataset/dataset_oleocelosis_yolo"
)

config = {
    "path": str(dataset_dir),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "sin oleocelosis",
        1: "con oleocelosis"
    }
}

yaml_correcto = dataset_dir / "data_smartfruit.yaml"

with open(yaml_correcto, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        config,
        f,
        allow_unicode=True,
        sort_keys=False
    )

print(yaml_correcto.read_text())
print("Archivo creado:", yaml_correcto)

from pathlib import Path
import torch
import yaml

yaml_path = Path(
    "/content/smartfruit_dataset/dataset_oleocelosis_yolo/data_smartfruit.yaml"
)

with open(yaml_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

root = Path(config["path"])

print("Configuración:", yaml_path.exists())
print("Clases:", config["names"])
print("Imágenes train:", len(list((root / "images/train").glob("*.jpg"))))
print("Imágenes val:", len(list((root / "images/val").glob("*.jpg"))))
print("Etiquetas train:", len(list((root / "labels/train").glob("*.txt"))))
print("Etiquetas val:", len(list((root / "labels/val").glob("*.txt"))))
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from ultralytics import YOLO

modelo = YOLO("yolo11n.pt")

resultados = modelo.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    seed=42,
    project="/content/smartfruit_runs",
    name="yolo_baseline",
    plots=True,
    save=True
)

from pathlib import Path
import shutil

origen = Path("/content/smartfruit_runs/yolo_baseline")
destino = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/resultados_yolo_baseline"
)

shutil.copytree(origen, destino, dirs_exist_ok=True)

print("Resultados copiados a:", destino)
print("Modelo best.pt:", (destino / "weights" / "best.pt").exists())

from IPython.display import display, Image
from pathlib import Path

carpeta = Path(
    "/content/smartfruit_runs/yolo_baseline"
)

graficos = [
    "results.png",
    "confusion_matrix.png",
    "PR_curve.png",
    "F1_curve.png",
    "labels.jpg"
]

for nombre in graficos:
    archivo = carpeta / nombre
    if archivo.exists():
        print(f"\n--- {nombre} ---")
        display(Image(filename=str(archivo)))
    else:
        print(f"No se encontró: {nombre}")

from pathlib import Path
import pandas as pd

carpeta = Path("/content/smartfruit_runs/yolo_baseline")

## 1. Métricas de las últimas épocas

In [ ]:
archivo = carpeta / "results.csv"

if archivo.exists():
    df = pd.read_csv(archivo)
    df.columns = df.columns.str.strip()

    columnas = [
        c for c in df.columns
        if any(x in c.lower() for x in [
            "epoch", "precision", "recall",
            "map50", "map75", "map50-95"
        ])
    ]

    print("ÚLTIMAS 5 ÉPOCAS:")
    print(df[columnas].tail().round(4).to_string(index=False))

## 2. Archivos gráficos disponibles

In [ ]:
print("\nGRÁFICOS GENERADOS:")
for archivo in sorted(carpeta.glob("*.png")):
    print("-", archivo.name)

## 3. Matriz de confusión normalizada, si existe

In [ ]:
from PIL import Image
import numpy as np

for nombre in ["confusion_matrix.png",
               "confusion_matrix_normalized.png"]:
    ruta = carpeta / nombre
    if ruta.exists():
        imagen = Image.open(ruta)
        print(f"\n{nombre}: tamaño {imagen.size}")
        display(imagen)

from ultralytics import YOLO
import numpy as np

modelo = YOLO(
    "/content/smartfruit_runs/yolo_baseline/weights/best.pt"
)

resultado = modelo.val(
    data="/content/smartfruit_dataset/dataset_oleocelosis_yolo/data_smartfruit.yaml",
    split="val",
    plots=True,
    verbose=False
)

cm = resultado.confusion_matrix

print("Clases:", modelo.names)
print("Tipo de matriz:", type(cm))
print("Atributos disponibles:", [
    a for a in dir(cm)
    if not a.startswith("_")
])

print("\nMatriz interna:")
print(repr(cm.matrix))

print("\nMatriz convertida a array:")
matriz = np.asarray(cm.matrix)
print(matriz)
print("Suma total:", matriz.sum())

print("\nMétricas obtenidas:")
print("Precisión global:", resultado.box.mp)
print("Recall global:", resultado.box.mr)
print("mAP50:", resultado.box.map50)
print("mAP50-95:", resultado.box.map)

from pathlib import Path
import random
import cv2
import matplotlib.pyplot as plt

base = Path(
    "/content/smartfruit_dataset/dataset_oleocelosis_yolo"
)
img_dir = base / "images" / "val"
lbl_dir = base / "labels" / "val"

archivos = list(img_dir.glob("*.jpg"))
seleccion = random.Random(42).sample(
    archivos, min(6, len(archivos))
)

nombres = {0: "Sin oleocelosis", 1: "Con oleocelosis"}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, img_path in zip(axes, seleccion):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    label_path = lbl_dir / f"{img_path.stem}.txt"

    if label_path.exists():
        for linea in label_path.read_text().splitlines():
            datos = linea.split()
            if len(datos) != 5:
                continue

            cls = int(datos[0])
            xc, yc, bw, bh = map(float, datos[1:])

            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)

            cv2.rectangle(
                img, (x1, y1), (x2, y2),
                (255, 0, 0), 2
            )
            cv2.putText(
                img, nombres.get(cls, str(cls)),
                (x1, max(y1 - 5, 15)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                (255, 0, 0), 1
            )

    ax.imshow(img)
    ax.set_title(img_path.name)
    ax.axis("off")

for ax in axes[len(seleccion):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## **BASELINE VERSIÓN 2 (RESNET18)**

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict

base = Path(
    "/content/smartfruit_dataset/dataset_oleocelosis_yolo"
)

clases = {
    0: "sin oleocelosis",
    1: "con oleocelosis"
}

resumen = Counter()
ejemplos = defaultdict(list)
errores = []
grupos_por_split = defaultdict(set)

for split in ["train", "val"]:
    img_dir = base / "images" / split
    lbl_dir = base / "labels" / split

    imagenes = [
        p for p in img_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    imagenes_por_stem = {p.stem: p for p in imagenes}
    etiquetas = {p.stem: p for p in lbl_dir.glob("*.txt")}

    sin_txt = set(imagenes_por_stem) - set(etiquetas)
    txt_sin_imagen = set(etiquetas) - set(imagenes_por_stem)

    print(f"\n{'=' * 45}")
    print(f"CONJUNTO: {split.upper()}")
    print("Imágenes:", len(imagenes))
    print("TXT:", len(etiquetas))
    print("Imágenes sin TXT:", len(sin_txt))
    print("TXT sin imagen:", len(txt_sin_imagen))

    for img in imagenes:
        txt = etiquetas.get(img.stem)

        if txt is None:
            errores.append((split, img.name, "Sin TXT"))
            continue

        ids_clase = []

        for linea in txt.read_text().splitlines():
            if not linea.strip():
                continue

            datos = linea.split()

            if len(datos) != 5:
                errores.append((split, img.name, "Formato incorrecto"))
                continue

            try:
                clase = int(datos[0])
            except ValueError:
                errores.append((split, img.name, "Clase inválida"))
                continue

            if clase not in clases:
                errores.append((split, img.name, f"Clase desconocida: {clase}"))
                continue

            ids_clase.append(clase)

        if not ids_clase:
            errores.append((split, img.name, "TXT vacío o sin etiquetas válidas"))
            continue

        clases_presentes = set(ids_clase)

        if len(clases_presentes) > 1:
            errores.append((split, img.name, "Mezcla de clases 0 y 1"))
            continue

        # La clase 1 indica lesión; la clase 0 indica fruto sano.
        etiqueta_fruto = 1 if 1 in clases_presentes else 0

        resumen[(split, etiqueta_fruto)] += 1
        grupos_por_split[split].add(img.stem.split("_L")[0])

        if len(ejemplos[(split, etiqueta_fruto)]) < 5:
            ejemplos[(split, etiqueta_fruto)].append(img.name)

print("\nRESUMEN DE ETIQUETAS POR IMAGEN")
for split in ["train", "val"]:
    for clase in [0, 1]:
        print(
            f"{split:5} | {clases[clase]:20} | "
            f"{resumen[(split, clase)]} imágenes"
        )

print("\nEJEMPLOS")
for clave, nombres in ejemplos.items():
    print(clave, ":", nombres)

print("\nERRORES ENCONTRADOS:", len(errores))
for error in errores[:20]:
    print(error)

solapamiento = (
    grupos_por_split["train"] & grupos_por_split["val"]
)
print("\nFotos originales compartidas entre train y val:",
      len(solapamiento))
if solapamiento:
    print("Ejemplos:", list(solapamiento)[:10])

from pathlib import Path
import shutil

base = Path(
    "/content/smartfruit_dataset/dataset_oleocelosis_yolo"
)

salida = Path(
    "/content/smartfruit_dataset/clasificacion_v1"
)

clases = {
    0: "sin_oleocelosis",
    1: "con_oleocelosis"
}

for split in ["train", "val"]:
    img_dir = base / "images" / split
    lbl_dir = base / "labels" / split

    for nombre in clases.values():
        (salida / split / nombre).mkdir(
            parents=True, exist_ok=True
        )

    for img in img_dir.iterdir():
        if img.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        txt = lbl_dir / f"{img.stem}.txt"

        if not txt.exists():
            raise ValueError(f"Falta TXT: {img.name}")

        ids = []
        for linea in txt.read_text().splitlines():
            if linea.strip():
                datos = linea.split()
                if len(datos) != 5:
                    raise ValueError(
                        f"Etiqueta inválida: {img.name}"
                    )
                ids.append(int(datos[0]))

        if not ids or not set(ids).issubset({0, 1}):
            raise ValueError(
                f"Etiquetas vacías o desconocidas: {img.name}"
            )

        if len(set(ids)) > 1:
            raise ValueError(
                f"Mezcla de clases en {img.name}"
            )

        clase = 1 if 1 in ids else 0
        destino = salida / split / clases[clase] / img.name

        shutil.copy2(img, destino)

print("Dataset de clasificación creado en:", salida)

for split in ["train", "val"]:
    print(f"\n{split.upper()}")
    for nombre in clases.values():
        carpeta = salida / split / nombre
        print(nombre, ":", len(list(carpeta.iterdir())))

from pathlib import Path
import torch
import pandas as pd
from torchvision import datasets, models, transforms
from torch import nn
from torch.utils.data import DataLoader
from torchvision.models import ResNet18_Weights

## 1. Rutas

In [ ]:
data_dir = Path(
    "/content/smartfruit_dataset/clasificacion_v1"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Dispositivo:", device)

## 2. Transformaciones

In [ ]:
transformaciones = {
    "train": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(
            brightness=0.10,
            contrast=0.10,
            saturation=0.10
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    "val": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
}

## 3. Cargar datasets

In [ ]:
datasets_img = {
    split: datasets.ImageFolder(
        data_dir / split,
        transform=transformaciones[split]
    )
    for split in ["train", "val"]
}

# Confirmar el índice correcto de cada clase
print("Mapeo de clases:", datasets_img["train"].class_to_idx)

idx_oleocelosis = datasets_img["train"].class_to_idx[
    "con_oleocelosis"
]

assert idx_oleocelosis == 0, (
    "Revisa el mapeo de clases antes de continuar."
)

generadores = {
    "train": DataLoader(
        datasets_img["train"],
        batch_size=32,
        shuffle=True,
        num_workers=2
    ),
    "val": DataLoader(
        datasets_img["val"],
        batch_size=32,
        shuffle=False,
        num_workers=2
    )
}

## 4. Modelo preentrenado

In [ ]:
modelo = models.resnet18(
    weights=ResNet18_Weights.DEFAULT
)
modelo.fc = nn.Linear(modelo.fc.in_features, 2)
modelo = modelo.to(device)

criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.AdamW(
    modelo.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

## 5. Entrenamiento y selección por recall de oleocelosis

In [ ]:
mejor_recall = -1
mejores_pesos = None
paciencia = 4
sin_mejora = 0
historial = []

for epoca in range(15):
    fila = {"epoca": epoca + 1}

    for fase in ["train", "val"]:
        modelo.train(fase == "train")

        perdidas = 0.0
        correctos = 0
        total = 0
        tp = fp = fn = tn = 0

        for imagenes, etiquetas in generadores[fase]:
            imagenes = imagenes.to(device)
            etiquetas = etiquetas.to(device)

            optimizador.zero_grad()

            with torch.set_grad_enabled(fase == "train"):
                salidas = modelo(imagenes)
                perdida = criterio(salidas, etiquetas)
                predicciones = salidas.argmax(dim=1)

                if fase == "train":
                    perdida.backward()
                    optimizador.step()

            perdidas += perdida.item() * imagenes.size(0)
            correctos += (
                predicciones == etiquetas
            ).sum().item()
            total += etiquetas.size(0)

            # Clase positiva: con oleocelosis (índice 0)
            tp += (
                (predicciones == 0) & (etiquetas == 0)
            ).sum().item()
            fp += (
                (predicciones == 0) & (etiquetas == 1)
            ).sum().item()
            fn += (
                (predicciones == 1) & (etiquetas == 0)
            ).sum().item()
            tn += (
                (predicciones == 1) & (etiquetas == 1)
            ).sum().item()

        loss_medio = perdidas / total
        accuracy = correctos / total
        recall = tp / (tp + fn) if tp + fn else 0
        precision = tp / (tp + fp) if tp + fp else 0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0
        )

        fila[f"{fase}_loss"] = loss_medio
        fila[f"{fase}_accuracy"] = accuracy
        fila[f"{fase}_recall_oleocelosis"] = recall
        fila[f"{fase}_precision_oleocelosis"] = precision
        fila[f"{fase}_f1_oleocelosis"] = f1

        if fase == "val":
            fila["val_TP"] = tp
            fila["val_FP"] = fp
            fila["val_FN"] = fn
            fila["val_TN"] = tn

    historial.append(fila)

    print(
        f"Época {epoca+1:02d} | "
        f"val_loss={fila['val_loss']:.4f} | "
        f"accuracy={fila['val_accuracy']:.3f} | "
        f"recall_oleocelosis="
        f"{fila['val_recall_oleocelosis']:.3f} | "
        f"precision_oleocelosis="
        f"{fila['val_precision_oleocelosis']:.3f} | "
        f"F1_oleocelosis="
        f"{fila['val_f1_oleocelosis']:.3f}"
    )

    if fila["val_recall_oleocelosis"] > mejor_recall:
        mejor_recall = fila["val_recall_oleocelosis"]
        mejores_pesos = {
            k: v.detach().cpu().clone()
            for k, v in modelo.state_dict().items()
        }
        sin_mejora = 0
    else:
        sin_mejora += 1

    if sin_mejora >= paciencia:
        print("Parada temprana.")
        break

## 6. Restaurar los mejores pesos de validación

In [ ]:
modelo.load_state_dict(mejores_pesos)

## 7. Guardar resultados en Drive

In [ ]:
destino = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "resultados_clasificador_baseline"
)
destino.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": modelo.state_dict(),
    "class_to_idx": datasets_img["train"].class_to_idx,
    "architecture": "ResNet18",
    "best_val_recall_oleocelosis": mejor_recall,
    "history": historial
}, destino / "resnet18_baseline.pth")

pd.DataFrame(historial).to_csv(
    destino / "historial_entrenamiento.csv",
    index=False
)

print("\nEntrenamiento finalizado.")
print("Mejor recall de validación:", round(mejor_recall, 4))
print("Resultados guardados en:", destino)

from pathlib import Path
import torch
import numpy as np
from torch import nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

data_dir = Path(
    "/content/smartfruit_dataset/clasificacion_v1/val"
)

archivo_modelo = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "resultados_clasificador_baseline/resnet18_baseline.pth"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

transformacion = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = datasets.ImageFolder(
    data_dir,
    transform=transformacion
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False
)

checkpoint = torch.load(
    archivo_modelo,
    map_location="cpu",
    weights_only=False
)

modelo = models.resnet18(weights=None)
modelo.fc = nn.Linear(modelo.fc.in_features, 2)
modelo.load_state_dict(checkpoint["model_state_dict"])
modelo = modelo.to(device)
modelo.eval()

y_real = []
y_pred = []

with torch.no_grad():
    for imagenes, etiquetas in loader:
        imagenes = imagenes.to(device)
        predicciones = modelo(imagenes).argmax(dim=1)

        y_real.extend(etiquetas.numpy())
        y_pred.extend(predicciones.cpu().numpy())

print("Orden de clases:", dataset.class_to_idx)
print("\nMatriz de confusión:")
print(confusion_matrix(y_real, y_pred, labels=[0, 1]))

print("\nReporte por clase:")
print(classification_report(
    y_real,
    y_pred,
    labels=[0, 1],
    target_names=[
        "Con oleocelosis",
        "Sin oleocelosis"
    ],
    digits=4
))

### **REVISIÓN DE IMÁGENES**

In [ ]:
from pathlib import Path
from collections import defaultdict

base = Path("/content/smartfruit_dataset/clasificacion_v1")

for particion in ["train", "val"]:
    ruta = base / particion
    grupos = defaultdict(int)
    total = 0

    for clase in ["con_oleocelosis", "sin_oleocelosis"]:
        carpeta = ruta / clase

        for archivo in carpeta.iterdir():
            if archivo.is_file():
                total += 1

                # Ejemplo: IMG_6705_L05.jpg -> IMG_6705
                grupo = archivo.stem.split("_L")[0]
                grupos[grupo] += 1

    print(f"\n{particion.upper()}")
    print("Imágenes:", total)
    print("Fotografías originales:", len(grupos))
    print("Recortes por fotografía:")
    print("  Mínimo:", min(grupos.values()))
    print("  Máximo:", max(grupos.values()))
    print("  Promedio:", round(total / len(grupos), 2))

grupos_train = {
    archivo.stem.split("_L")[0]
    for clase in ["con_oleocelosis", "sin_oleocelosis"]
    for archivo in (base / "train" / clase).iterdir()
    if archivo.is_file()
}

grupos_val = {
    archivo.stem.split("_L")[0]
    for clase in ["con_oleocelosis", "sin_oleocelosis"]
    for archivo in (base / "val" / clase).iterdir()
    if archivo.is_file()
}

print("\nGrupos compartidos entre train y val:",
      len(grupos_train & grupos_val))

from pathlib import Path
import shutil
import numpy as np

origen = Path("/content/smartfruit_dataset/clasificacion_v1")
destino = Path("/content/smartfruit_dataset/clasificacion_v2")

clases = ["con_oleocelosis", "sin_oleocelosis"]
rng = np.random.default_rng(42)

# Recopilar imágenes y etiquetas del train actual
archivos = []
etiquetas = []
grupos = []

for clase_id, clase in enumerate(clases):
    for archivo in (origen / "train" / clase).iterdir():
        if archivo.is_file():
            archivos.append(archivo)
            etiquetas.append(clase_id)
            grupos.append(archivo.stem.split("_L")[0])

etiquetas = np.array(etiquetas)
grupos = np.array(grupos)

# Un grupo representa una fotografía original.
# Cada fotografía se asigna íntegramente a train o test.
grupos_unicos = np.unique(grupos)
rng.shuffle(grupos_unicos)

grupos_test = set()
total_test = 0
objetivo_test = round(0.20 * len(archivos))

# Selección aproximada del 20% de las imágenes,
# procurando que ambas clases estén representadas.
for grupo in grupos_unicos:
    indices_grupo = np.where(grupos == grupo)[0]
    cantidad_grupo = len(indices_grupo)

    if total_test < objetivo_test:
        grupos_test.add(grupo)
        total_test += cantidad_grupo

indices_test = [
    i for i, grupo in enumerate(grupos)
    if grupo in grupos_test
]
indices_train = [
    i for i, grupo in enumerate(grupos)
    if grupo not in grupos_test
]

# Preparar una nueva estructura, sin borrar el original.
if destino.exists():
    raise FileExistsError(
        f"Ya existe {destino}. No se sobrescribió. "
        "Usa otro nombre de carpeta si necesitas repetir."
    )

for particion in ["train", "val", "test"]:
    for clase in clases:
        (destino / particion / clase).mkdir(
            parents=True, exist_ok=True
        )

# Copiar train y test
for particion, indices in [
    ("train", indices_train),
    ("test", indices_test)
]:
    for i in indices:
        clase = clases[int(etiquetas[i])]
        shutil.copy2(
            archivos[i],
            destino / particion / clase / archivos[i].name
        )

# Copiar validación sin modificarla
for clase in clases:
    for archivo in (origen / "val" / clase).iterdir():
        if archivo.is_file():
            shutil.copy2(
                archivo,
                destino / "val" / clase / archivo.name
            )

# Revisar cantidades y fotografías originales
grupos_particion = {}

for particion in ["train", "val", "test"]:
    print(f"\n{particion.upper()}")
    grupos_particion[particion] = set()

    for clase in clases:
        carpeta = destino / particion / clase
        imagenes = [f for f in carpeta.iterdir() if f.is_file()]

        print(f"  {clase}: {len(imagenes)} imágenes")

        grupos_particion[particion].update(
            f.stem.split("_L")[0] for f in imagenes
        )

    print(
        "  Fotografías originales:",
        len(grupos_particion[particion])
    )

print("\nSOLAPAMIENTO ENTRE FOTOGRAFÍAS")
for a, b in [
    ("train", "val"),
    ("train", "test"),
    ("val", "test")
]:
    print(
        f"{a} vs {b}:",
        len(grupos_particion[a] & grupos_particion[b]),
        "grupos compartidos"
    )

print("\nDataset creado en:", destino)

### **REENTRENAR**

In [ ]:
from pathlib import Path
import copy
import csv
import numpy as np
import torch

from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import precision_score, recall_score, f1_score

## 1. Configuración

In [ ]:
base = Path("/content/smartfruit_dataset/clasificacion_v2")

salida = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "resultados_clasificador_test"
)
salida.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

# Mantenemos el mismo preprocesamiento del baseline
transformacion = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_ds = datasets.ImageFolder(
    base / "train", transform=transformacion
)
val_ds = datasets.ImageFolder(
    base / "val", transform=transformacion
)

print("Clases:", train_ds.class_to_idx)
assert train_ds.class_to_idx == {
    "con_oleocelosis": 0,
    "sin_oleocelosis": 1
}, "Revisa el orden de las clases antes de continuar."

train_loader = DataLoader(
    train_ds, batch_size=32, shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_ds, batch_size=32, shuffle=False, num_workers=2
)

## 2. Modelo preentrenado

In [ ]:
modelo = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)
modelo.fc = nn.Linear(modelo.fc.in_features, 2)
modelo = modelo.to(device)

criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(
    modelo.parameters(), lr=0.0001
)

## 3. Entrenamiento y validación

In [ ]:
max_epocas = 15
paciencia = 5

mejor_recall = -1.0
mejor_f1 = -1.0
epoca_mejor = 0
sin_mejora = 0
mejores_pesos = None
historial = []

for epoca in range(1, max_epocas + 1):

    modelo.train()
    perdida_train = 0.0

    for imagenes, etiquetas in train_loader:
        imagenes = imagenes.to(device)
        etiquetas = etiquetas.to(device)

        optimizador.zero_grad()
        logits = modelo(imagenes)
        perdida = criterio(logits, etiquetas)
        perdida.backward()
        optimizador.step()

        perdida_train += perdida.item() * imagenes.size(0)

    perdida_train /= len(train_ds)

    # Validación: no se entrena con estas imágenes
    modelo.eval()
    perdida_val = 0.0
    reales, predichas = [], []

    with torch.no_grad():
        for imagenes, etiquetas in val_loader:
            imagenes = imagenes.to(device)
            etiquetas_device = etiquetas.to(device)

            logits = modelo(imagenes)
            perdida = criterio(logits, etiquetas_device)

            perdida_val += perdida.item() * imagenes.size(0)
            predicciones = logits.argmax(dim=1)

            reales.extend(etiquetas.numpy())
            predichas.extend(predicciones.cpu().numpy())

    perdida_val /= len(val_ds)

    # Clase positiva: con oleocelosis (índice 0)
    recall = recall_score(
        reales, predichas, pos_label=0, zero_division=0
    )
    precision = precision_score(
        reales, predichas, pos_label=0, zero_division=0
    )
    f1 = f1_score(
        reales, predichas, pos_label=0, zero_division=0
    )
    accuracy = np.mean(np.array(reales) == np.array(predichas))

    print(
        f"Época {epoca:02d}/{max_epocas} | "
        f"loss_train={perdida_train:.4f} | "
        f"loss_val={perdida_val:.4f} | "
        f"accuracy={accuracy:.4f} | "
        f"recall={recall:.4f} | "
        f"precision={precision:.4f} | F1={f1:.4f}"
    )

    historial.append({
        "epoca": epoca,
        "loss_train": perdida_train,
        "loss_val": perdida_val,
        "accuracy_val": accuracy,
        "recall_oleocelosis": recall,
        "precision_oleocelosis": precision,
        "f1_oleocelosis": f1
    })

    # Priorizar recall; usar F1 para desempatar
    mejora = (
        recall > mejor_recall + 1e-4
        or (
            abs(recall - mejor_recall) <= 1e-4
            and f1 > mejor_f1 + 1e-4
        )
    )

    if mejora:
        mejor_recall = recall
        mejor_f1 = f1
        epoca_mejor = epoca
        mejores_pesos = copy.deepcopy(modelo.state_dict())
        sin_mejora = 0
    else:
        sin_mejora += 1

    if sin_mejora >= paciencia:
        print("Parada temprana.")
        break

## 4. Guardar el mejor modelo y el historial

In [ ]:
if mejores_pesos is None:
    raise RuntimeError("No se guardaron pesos del modelo.")

archivo_modelo = salida / "resnet18_reentrenado.pth"
torch.save({
    "model_state_dict": mejores_pesos,
    "class_to_idx": train_ds.class_to_idx,
    "epoca_mejor": epoca_mejor,
    "mejor_recall_val": mejor_recall,
    "mejor_f1_val": mejor_f1
}, archivo_modelo)

archivo_historial = salida / "historial_entrenamiento.csv"

with open(archivo_historial, "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(
        f, fieldnames=historial[0].keys()
    )
    escritor.writeheader()
    escritor.writerows(historial)

print("\nEntrenamiento terminado.")
print("Mejor época:", epoca_mejor)
print(f"Mejor recall de validación: {mejor_recall:.4f}")
print(f"F1 de validación: {mejor_f1:.4f}")
print("Modelo guardado en:", archivo_modelo)
print("Historial guardado en:", archivo_historial)

from pathlib import Path
import torch
from torch import nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

base = Path("/content/smartfruit_dataset/clasificacion_v2")
archivo_modelo = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "resultados_clasificador_test/resnet18_reentrenado.pth"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transformacion = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_ds = datasets.ImageFolder(
    base / "test",
    transform=transformacion
)

test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False
)

checkpoint = torch.load(
    archivo_modelo,
    map_location="cpu",
    weights_only=False
)

modelo = models.resnet18(weights=None)
modelo.fc = nn.Linear(modelo.fc.in_features, 2)
modelo.load_state_dict(checkpoint["model_state_dict"])
modelo = modelo.to(device)
modelo.eval()

y_real = []
y_pred = []

with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        imagenes = imagenes.to(device)

        logits = modelo(imagenes)
        predicciones = logits.argmax(dim=1)

        y_real.extend(etiquetas.numpy())
        y_pred.extend(predicciones.cpu().numpy())

print("Orden de clases:", test_ds.class_to_idx)
print("\nCantidad de imágenes:", len(test_ds))
print("\nAccuracy:", round(accuracy_score(y_real, y_pred), 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_real, y_pred, labels=[0, 1]))

print("\nReporte por clase:")
print(classification_report(
    y_real,
    y_pred,
    labels=[0, 1],
    target_names=["Con oleocelosis", "Sin oleocelosis"],
    digits=4,
    zero_division=0
))

### **REENTRENAMIENTO BASELINE V1 (YOLO)**

In [ ]:
from pathlib import Path
import shutil
import yaml

# Dataset YOLO original
origen = Path(
    "/content/smartfruit_dataset/dataset_oleocelosis_yolo"
)

# Particiones ya preparadas para ResNet18
base_clasificacion = Path(
    "/content/smartfruit_dataset/clasificacion_v2"
)

# Nueva versión YOLO; no modifica el dataset original
destino = Path(
    "/content/smartfruit_dataset/dataset_oleocelosis_yolo_v2"
)

if destino.exists():
    raise FileExistsError(
        f"Ya existe {destino}. No se sobrescribió."
    )

particiones = ["train", "val", "test"]
grupos_por_particion = {}

# Identificar las fotografías originales de cada partición
for particion in particiones:
    grupos = set()

    for clase in ["con_oleocelosis", "sin_oleocelosis"]:
        carpeta = base_clasificacion / particion / clase

        for archivo in carpeta.iterdir():
            if archivo.is_file():
                grupos.add(archivo.stem.split("_L")[0])

    grupos_por_particion[particion] = grupos

# Buscar las imágenes originales del dataset YOLO
imagenes_originales = {}

for particion_original in ["train", "val"]:
    carpeta = origen / "images" / particion_original

    for archivo in carpeta.iterdir():
        if archivo.is_file():
            imagenes_originales[archivo.name] = archivo

# Crear particiones YOLO con los mismos grupos
for particion in particiones:
    (destino / "images" / particion).mkdir(
        parents=True, exist_ok=True
    )
    (destino / "labels" / particion).mkdir(
        parents=True, exist_ok=True
    )

    grupos_objetivo = grupos_por_particion[particion]
    copiados = 0
    faltantes = []

    for nombre, archivo in imagenes_originales.items():
        grupo = archivo.stem.split("_L")[0]

        if grupo not in grupos_objetivo:
            continue

        etiqueta = (
            origen / "labels" /
            ("train" if (origen / "labels" / "train" /
                         f"{archivo.stem}.txt").exists()
             else "val") /
            f"{archivo.stem}.txt"
        )

        if not etiqueta.exists():
            faltantes.append(nombre)
            continue

        shutil.copy2(
            archivo, destino / "images" / particion / nombre
        )
        shutil.copy2(
            etiqueta,
            destino / "labels" / particion / etiqueta.name
        )
        copiados += 1

    print(
        f"{particion.upper()}: {copiados} imágenes y etiquetas copiadas"
    )

    if faltantes:
        print("  ADVERTENCIA: etiquetas faltantes:", len(faltantes))
        print("  Ejemplos:", faltantes[:5])

# Crear configuración YOLO
config = {
    "path": str(destino),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "sin oleocelosis",
        1: "con oleocelosis"
    }
}

archivo_yaml = destino / "data.yaml"

with open(archivo_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        config,
        f,
        allow_unicode=True,
        sort_keys=False
    )

print("\nConfiguración creada:", archivo_yaml)

from pathlib import Path
from collections import Counter
import yaml

base = Path("/content/smartfruit_dataset/dataset_oleocelosis_yolo_v2")
base_clas = Path("/content/smartfruit_dataset/clasificacion_v2")

with open(base / "data.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Clases YOLO:", config["names"])

grupos = {}

for split in ["train", "val", "test"]:
    imgs = list((base / "images" / split).glob("*"))
    imgs = [p for p in imgs if p.is_file()]
    labels_dir = base / "labels" / split

    nombres_img = {p.stem for p in imgs}
    nombres_lbl = {p.stem for p in labels_dir.glob("*.txt")}

    print(f"\n{split.upper()}")
    print("Imágenes:", len(imgs))
    print("Archivos de etiquetas:", len(nombres_lbl))
    print("Sin etiqueta:", len(nombres_img - nombres_lbl))
    print("Etiqueta sin imagen:", len(nombres_lbl - nombres_img))

    grupos[split] = {p.stem.split("_L")[0] for p in imgs}

    conteo = Counter()
    etiquetas_vacias = 0
    etiquetas_invalidas = []

    for imagen in imgs:
        archivo = labels_dir / f"{imagen.stem}.txt"

        if not archivo.exists():
            continue

        lineas = archivo.read_text(encoding="utf-8").strip().splitlines()

        if not lineas or lineas == [""]:
            etiquetas_vacias += 1
            continue

        for linea in lineas:
            partes = linea.split()
            try:
                clase = int(partes[0])
                coords = [float(x) for x in partes[1:]]

                if clase not in [0, 1] or len(coords) != 4:
                    raise ValueError()

                if not all(0 <= x <= 1 for x in coords):
                    raise ValueError()

                conteo[clase] += 1

            except ValueError:
                etiquetas_invalidas.append((archivo.name, linea))

    print("Cajas por clase:", dict(conteo))
    print("Etiquetas vacías:", etiquetas_vacias)
    print("Líneas inválidas:", len(etiquetas_invalidas))

    # Confirmar que los mismos recortes estén en la partición de clasificación
    nombres_clas = {
        p.stem
        for clase in ["con_oleocelosis", "sin_oleocelosis"]
        for p in (base_clas / split / clase).iterdir()
        if p.is_file()
    }

    print("Mismos nombres que clasificación:", nombres_img == nombres_clas)

print("\nSOLAPAMIENTO DE FOTOGRAFÍAS")
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    print(f"{a} vs {b}:", len(grupos[a] & grupos[b]))

from pathlib import Path
from ultralytics import YOLO
import torch

# Dataset revisado
archivo_yaml = Path(
    "/content/smartfruit_dataset/dataset_oleocelosis_yolo_v2/data.yaml"
)

# Carpeta nueva: no sobrescribe el baseline anterior
proyecto = "/content/smartfruit_runs"
nombre = "yolo_reentrenado_test"

assert archivo_yaml.exists(), "No se encuentra data.yaml"

# Verificar GPU
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Inicializar YOLO11n con pesos preentrenados
modelo = YOLO("yolo11n.pt")

# Entrenar
resultados = modelo.train(
    data=str(archivo_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    seed=42,
    deterministic=True,
    pretrained=True,
    workers=2,
    project=proyecto,
    name=nombre,
    exist_ok=False,
    plots=True,
    verbose=True
)

print("\nEntrenamiento finalizado.")
print("Mejores pesos:", f"{proyecto}/{nombre}/weights/best.pt")
print("Últimos pesos:", f"{proyecto}/{nombre}/weights/last.pt")
print("Resultados y gráficos:", f"{proyecto}/{nombre}")

## SMARTFRUIT — RECUPERAR BASELINE YOLO11n DESDE GOOGLE DRIVE

In [ ]:
from pathlib import Path
import subprocess
import sys

## 1. Instalar Ultralytics solo si no está disponible

In [ ]:
try:
    import ultralytics
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "ultralytics"
    ])

from ultralytics import YOLO

## 2. Conectar Google Drive si todavía no está montado

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Definir rutas persistentes

In [ ]:
carpeta_drive = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "resultados_yolo_reentrenado_test"
)

pesos = carpeta_drive / "best.pt"
carpeta_evaluacion = carpeta_drive / "evaluacion_test"

## 4. Verificar que el modelo esté guardado

In [ ]:
if not pesos.exists():
    raise FileNotFoundError(
        f"No se encontró el modelo guardado en Drive: {pesos}\n"
        "Revisa la ruta antes de continuar. No se volverá a entrenar."
    )

print("Modelo encontrado:", pesos)
print("Tamaño del modelo (MB):", round(pesos.stat().st_size / (1024**2), 2))

## 5. Recuperar los resultados de la evaluación anterior

In [ ]:
if carpeta_evaluacion.exists():
    archivos = sorted(
        p for p in carpeta_evaluacion.rglob("*") if p.is_file()
    )
    print("\nArchivos de evaluación encontrados:", len(archivos))
    for archivo in archivos:
        print(" -", archivo.relative_to(carpeta_drive))
else:
    print("\nNo se encontró la carpeta de evaluación en Drive.")

## 6. Cargar el modelo guardado

In [ ]:
model = YOLO(str(pesos))

print("\nYOLO11n cargado correctamente.")
print("Clases:", model.names)

## 7. Comprobar si el dataset TEST sigue disponible

In [ ]:
dataset_test = Path(
    "/content/smartfruit_dataset/"
    "dataset_oleocelosis_yolo_v2/images/test"
)

if dataset_test.exists():
    imagenes_test = sorted(
        p for p in dataset_test.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    )
    print("\nDataset TEST disponible:", len(imagenes_test), "imágenes")
else:
    imagenes_test = []
    print(
        "\nEl dataset TEST no está disponible en esta sesión de Colab."
        "\nNo pasa nada: el modelo y los resultados guardados en Drive "
        "se mantienen."
    )

print("\nRecuperación finalizada. No se volvió a entrenar ni evaluar TEST.")

from pathlib import Path
import zipfile

# Carpeta persistente de SmartFruit en Drive
proyecto = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO"
)

# Destino temporal del dataset
destino = Path("/content/smartfruit_dataset/dataset_oleocelosis_yolo_v2")
destino.mkdir(parents=True, exist_ok=True)

# Si TEST ya existe, no extraemos nada
test_dir = destino / "images" / "test"

if test_dir.exists() and any(test_dir.iterdir()):
    print("El dataset TEST ya está disponible.")
else:
    # Buscar archivos ZIP del dataset en la carpeta del proyecto
    zips = list(proyecto.glob("*.zip"))

    print("ZIP encontrados:")
    for z in zips:
        print(" -", z.name)

    # Buscar el ZIP original de YOLO
    candidatos = [
        z for z in zips
        if "dataset_oleocelosis_yolo" in z.name.lower()
    ]

    if not candidatos:
        print(
            "\nNo encontré un ZIP cuyo nombre contenga "
            "'dataset_oleocelosis_yolo'."
            "\nNo se extrajo ningún archivo. Revisaremos dónde "
            "guardaste el dataset."
        )
    else:
        zip_path = candidatos[0]
        print("\nZIP seleccionado:", zip_path.name)

        with zipfile.ZipFile(zip_path, "r") as zf:
            nombres = zf.namelist()

            # Buscar la carpeta que contiene images/test
            prefijos = set()
            for nombre in nombres:
                partes = nombre.strip("/").split("/")
                for i in range(len(partes) - 1):
                    if partes[i] == "images" and partes[i + 1] == "test":
                        prefijos.add("/".join(partes[:i]))

            if not prefijos:
                print(
                    "El ZIP no contiene una carpeta images/test. "
                    "No se extrajo nada."
                )
            else:
                prefijo = sorted(prefijos, key=len)[0]
                seleccionados = [
                    n for n in nombres
                    if n.startswith(prefijo + "/")
                    and (
                        n.startswith(prefijo + "/images/test/")
                        or n.startswith(prefijo + "/labels/test/")
                    )
                ]

                for nombre in seleccionados:
                    zf.extract(nombre, "/content")

                print("Archivos extraídos:", len(seleccionados))

# Comprobar resultado
if test_dir.exists():
    imagenes = [
        p for p in test_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ]
    print("Imágenes TEST disponibles:", len(imagenes))

from pathlib import Path

proyecto = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO"
)

print("BUSCANDO DATASETS Y ARCHIVOS ZIP EN SMARTFRUIT\n")

for elemento in proyecto.rglob("*"):
    if elemento.is_file() and (
        elemento.suffix.lower() == ".zip"
        or elemento.name == "data.yaml"
    ):
        print(elemento)

    elif elemento.is_dir() and elemento.name in [
        "dataset_oleocelosis_yolo_v2",
        "test",
        "images",
        "labels"
    ]:
        print("[CARPETA]", elemento)

from pathlib import Path
import zipfile

zip_path = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "dataset_oleocelosis_yolo.zip"
)

destino = Path("/content/smartfruit_dataset")

with zipfile.ZipFile(zip_path, "r") as zf:
    nombres = zf.namelist()

    print("Primeros archivos dentro del ZIP:")
    for nombre in nombres[:25]:
        print(nombre)

    # Extraer el dataset original sin borrar archivos existentes
    zf.extractall(destino)

print("\nExtracción terminada.")
print("Contenido de /content/smartfruit_dataset:")
for elemento in sorted(destino.iterdir()):
    print(
        "[CARPETA]" if elemento.is_dir() else "[ARCHIVO]",
        elemento.name
    )

from pathlib import Path

rutas = [
    Path("/content/smartfruit_dataset/dataset_oleocelosis_yolo"),
    Path("/content/smartfruit_dataset/dataset_oleocelosis_yolo_v2"),
]

for ruta in rutas:
    print(f"\n{'=' * 65}")
    print("CARPETA:", ruta)
    print("Existe:", ruta.exists())

    if ruta.exists():
        for subcarpeta in ["images/train", "images/val", "images/test",
                           "labels/train", "labels/val", "labels/test"]:
            p = ruta / subcarpeta
            if p.exists():
                archivos = [f for f in p.iterdir() if f.is_file()]
                print(f"{subcarpeta}: {len(archivos)} archivos")
            else:
                print(f"{subcarpeta}: NO EXISTE")

        yaml = ruta / "data.yaml"
        print("data.yaml:", yaml.exists())

from pathlib import Path

proyecto = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO"
)

print("Buscando archivos que permitan recuperar la partición original...\n")

extensiones = {".csv", ".json", ".txt", ".yaml", ".yml", ".xlsx", ".pkl"}

for archivo in proyecto.rglob("*"):
    if (
        archivo.is_file()
        and archivo.suffix.lower() in extensiones
        and any(
            palabra in archivo.name.lower()
            for palabra in [
                "split", "partition", "manifest", "group",
                "dataset", "test", "train", "val", "repart"
            ]
        )
    ):
        print(archivo)

from pathlib import Path

proyecto = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO"
)

carpeta_eval = proyecto / "resultados_yolo_reentrenado_test" / "evaluacion_test"

print("=== ARCHIVOS DE EVALUACIÓN ===")
for p in sorted(carpeta_eval.rglob("*")):
    if p.is_file():
        print(p.name, "|", round(p.stat().st_size / 1024, 1), "KB")

print("\n=== EJEMPLOS DEL DATASET ORIGINAL ===")
base = Path("/content/smartfruit_dataset/dataset_oleocelosis_yolo")

for particion in ["train", "val"]:
    carpeta = base / "images" / particion
    nombres = sorted(p.name for p in carpeta.glob("*") if p.is_file())
    print(f"\n{particion}: {len(nombres)} imágenes")
    print("Primeras 10:", nombres[:10])

print("\n=== CONFIGURACIÓN ORIGINAL ===")
print((base / "data.yaml").read_text())

from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import random

# Modelo persistente recuperado desde Drive
modelo_path = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "resultados_yolo_reentrenado_test/best.pt"
)

# Dataset original disponible en Colab
base = Path("/content/smartfruit_dataset/dataset_oleocelosis_yolo")
carpeta_train = base / "images" / "train"
carpeta_val = base / "images" / "val"

# Usaremos imágenes de validación para inspección exploratoria
imagenes = sorted(carpeta_val.glob("*.jpg"))

if not imagenes:
    raise FileNotFoundError(
        f"No se encontraron imágenes JPG en {carpeta_val}"
    )

modelo = YOLO(str(modelo_path))

# Muestra reproducible de seis imágenes
random.seed(42)
muestras = random.sample(imagenes, min(6, len(imagenes)))

for ruta in muestras:
    resultado = modelo.predict(
        source=str(ruta),
        imgsz=640,
        conf=0.25,
        verbose=False
    )[0]

    imagen = resultado.plot()

    plt.figure(figsize=(8, 6))
    plt.imshow(imagen[:, :, ::-1])
    plt.title(f"Revisión exploratoria: {ruta.name}")
    plt.axis("off")
    plt.show()

    print(f"Imagen: {ruta.name}")
    if len(resultado.boxes):
        for caja in resultado.boxes:
            clase = int(caja.cls.item())
            confianza = float(caja.conf.item())
            print(f"  {modelo.names[clase]} — confianza {confianza:.3f}")
    else:
        print("  No se detectaron cajas.")

    print("-" * 45)

from pathlib import Path
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import matplotlib.patches as patches

base = Path("/content/smartfruit_dataset/dataset_oleocelosis_yolo")
imagenes_dir = base / "images" / "val"
etiquetas_dir = base / "labels" / "val"

nombres = [
    "IMG_6565_L11.jpg",
    "IMG_6437_L10.jpg",
    "IMG_6421_L01.jpg",
    "IMG_6588_L05.jpg",
    "IMG_6465_L09.jpg",
    "IMG_6455_L06.jpg",
]

# Colores para diferenciar referencia y predicción
colores = {
    0: "lime",
    1: "red",
}

for nombre in nombres:
    ruta_img = imagenes_dir / nombre
    ruta_label = etiquetas_dir / f"{Path(nombre).stem}.txt"

    if not ruta_img.exists():
        print("No se encontró:", ruta_img)
        continue

    imagen = PILImage.open(ruta_img).convert("RGB")
    ancho, alto = imagen.size

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.imshow(imagen)

    # Anotaciones de referencia: línea continua
    if ruta_label.exists():
        for linea in ruta_label.read_text().splitlines():
            datos = linea.split()
            if len(datos) < 5:
                continue

            clase = int(float(datos[0]))
            xc, yc, w, h = map(float, datos[1:5])

            x = (xc - w / 2) * ancho
            y = (yc - h / 2) * alto
            w *= ancho
            h *= alto

            rect = patches.Rectangle(
                (x, y), w, h,
                linewidth=2,
                edgecolor=colores.get(clase, "yellow"),
                facecolor="none",
                linestyle="-"
            )
            ax.add_patch(rect)
            ax.text(
                x, max(0, y - 5),
                f"REAL: {clase}",
                color="white",
                fontsize=8,
                bbox=dict(facecolor=colores.get(clase, "yellow"), alpha=0.8)
            )

    # Predicciones del modelo: línea discontinua
    resultado = model.predict(
        source=str(ruta_img),
        imgsz=640,
        conf=0.25,
        verbose=False
    )[0]

    for caja in resultado.boxes:
        clase = int(caja.cls.item())
        confianza = float(caja.conf.item())
        x1, y1, x2, y2 = caja.xyxy[0].tolist()

        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2,
            edgecolor=colores.get(clase, "yellow"),
            facecolor="none",
            linestyle="--"
        )
        ax.add_patch(rect)
        ax.text(
            x1, min(alto - 5, y2 + 12),
            f"PRED: {clase} ({confianza:.2f})",
            color="white",
            fontsize=8,
            bbox=dict(facecolor=colores.get(clase, "yellow"), alpha=0.8)
        )

    ax.set_title(f"{nombre}\nContinua = real | Discontinua = predicción")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

## SMARTFRUIT — CREACIÓN DE DATASET REPRODUCIBLE V3

In [ ]:
# Train/Test por grupos de fotografía + Val original
# Guarda dataset, manifiesto y configuración en Google Drive

In [ ]:
from pathlib import Path
from collections import Counter
import shutil
import random
import json
import csv
import hashlib

import numpy as np
import pandas as pd
from google.colab import drive

## 1. Montar Google Drive

In [ ]:
drive.mount("/content/drive")

## 2. Localizar el dataset original

In [ ]:
candidatos = [
    Path("/content/smartfruit_dataset/dataset_oleocelosis_yolo"),
    Path("/content/dataset_oleocelosis_yolo"),
]

SOURCE = next(
    (p for p in candidatos
     if (p / "images/train").exists()
     and (p / "labels/train").exists()
     and (p / "images/val").exists()
     and (p / "labels/val").exists()),
    None
)

if SOURCE is None:
    raise FileNotFoundError(
        "No encuentro el dataset original. Verifica que el ZIP "
        "esté extraído en /content/smartfruit_dataset/."
    )

print("Dataset original:", SOURCE)

## 3. Destino persistente: no sobrescribe versiones anteriores

In [ ]:
DEST = (
    Path("/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
         "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
         "dataset_oleocelosis_yolo_v3_reproducible")
)

if DEST.exists():
    raise FileExistsError(
        f"Ya existe {DEST}. No lo sobrescribiré para proteger "
        "los datos. Si deseas repetir el proceso, usa otra "
        "carpeta con un nombre de versión diferente."
    )

## 4. Reunir imágenes y comprobar que cada una tenga etiqueta

In [ ]:
def obtener_pares(particion):
    img_dir = SOURCE / "images" / particion
    lab_dir = SOURCE / "labels" / particion

    imagenes = sorted(
        p for p in img_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]
    )

    pares = []
    errores = []

    for img in imagenes:
        label = lab_dir / f"{img.stem}.txt"
        if not label.exists():
            errores.append(img.name)
        else:
            pares.append((img, label))

    if errores:
        raise ValueError(
            f"{particion}: {len(errores)} imágenes sin etiqueta. "
            f"Ejemplos: {errores[:5]}"
        )

    return pares

train_original = obtener_pares("train")
val_original = obtener_pares("val")

print("Imágenes originales train:", len(train_original))
print("Imágenes originales val:", len(val_original))

## 5. Identificar grupos por fotografía de origen

In [ ]:
# Ejemplo: IMG_6705_L05.jpg -> grupo IMG_6705
def grupo_origen(img):
    return img.stem.split("_L")[0]

grupos_train = sorted({
    grupo_origen(img) for img, _ in train_original
})
grupos_val = {
    grupo_origen(img) for img, _ in val_original
}

solapamiento = set(grupos_train) & grupos_val

if solapamiento:
    raise ValueError(
        "Hay grupos de fotografías compartidos entre el train "
        f"y el val originales. Ejemplos: {list(solapamiento)[:5]}"
    )

## 6. Separar grupos de train en train nuevo y test nuevo

In [ ]:
SEED = 42
FRACCION_TEST = 0.20

rng = np.random.default_rng(SEED)
grupos_barajados = np.array(grupos_train, dtype=object)
rng.shuffle(grupos_barajados)

n_test = max(1, round(len(grupos_barajados) * FRACCION_TEST))
grupos_test = set(grupos_barajados[:n_test].tolist())
grupos_train_nuevo = set(grupos_barajados[n_test:].tolist())

pares_train = [
    par for par in train_original
    if grupo_origen(par[0]) in grupos_train_nuevo
]
pares_test = [
    par for par in train_original
    if grupo_origen(par[0]) in grupos_test
]
pares_val = val_original

assert grupos_train_nuevo.isdisjoint(grupos_test)
assert grupos_train_nuevo.isdisjoint(grupos_val)
assert grupos_test.isdisjoint(grupos_val)

print("Grupos train:", len(grupos_train_nuevo))
print("Grupos val:", len(grupos_val))
print("Grupos test:", len(grupos_test))

## 7. Crear carpetas y copiar archivos

In [ ]:
for split in ["train", "val", "test"]:
    (DEST / "images" / split).mkdir(parents=True, exist_ok=True)
    (DEST / "labels" / split).mkdir(parents=True, exist_ok=True)

particiones = {
    "train": pares_train,
    "val": pares_val,
    "test": pares_test,
}

manifiesto = []

for split, pares in particiones.items():
    for img, label in pares:
        destino_img = DEST / "images" / split / img.name
        destino_label = DEST / "labels" / split / label.name

        shutil.copy2(img, destino_img)
        shutil.copy2(label, destino_label)

        # Registrar clase(s) presentes en las anotaciones YOLO
        clases = []
        with open(label, "r", encoding="utf-8") as f:
            for linea in f:
                campos = linea.strip().split()
                if campos:
                    clases.append(int(float(campos[0])))

        manifiesto.append({
            "imagen": img.name,
            "grupo_origen": grupo_origen(img),
            "particion": split,
            "ruta_imagen": str(destino_img.relative_to(DEST)),
            "ruta_etiqueta": str(destino_label.relative_to(DEST)),
            "clases_anotadas": ",".join(map(str, sorted(set(clases)))),
            "numero_cajas": len(clases),
            "sha256_imagen": hashlib.sha256(img.read_bytes()).hexdigest(),
            "sha256_etiqueta": hashlib.sha256(label.read_bytes()).hexdigest(),
        })

## 8. Guardar manifiesto

In [ ]:
df = pd.DataFrame(manifiesto)
ruta_manifest = DEST / "manifest_particiones.csv"
df.to_csv(ruta_manifest, index=False, encoding="utf-8-sig")

## 9. Guardar configuración de YOLO

In [ ]:
yaml_text = f"""path: {DEST}
train: images/train
val: images/val
test: images/test
names:
  0: sin oleocelosis
  1: con oleocelosis
"""
(DEST / "data.yaml").write_text(yaml_text, encoding="utf-8")

## 10. Guardar resumen reproducible

In [ ]:
resumen = {
    "proyecto": "SmartFruit",
    "version_dataset": "v3_reproducible",
    "semilla": SEED,
    "metodo": (
        "Separación por grupos de fotografía de origen. "
        "El train original se divide por grupos en train/test; "
        "el val original se conserva."
    ),
    "fraccion_grupos_test_sobre_train_original": FRACCION_TEST,
    "conteos_imagenes": {
        split: len(pares) for split, pares in particiones.items()
    },
    "conteos_grupos": {
        "train": len(grupos_train_nuevo),
        "val": len(grupos_val),
        "test": len(grupos_test),
    },
    "clases": {
        "0": "sin oleocelosis",
        "1": "con oleocelosis",
    },
    "nota": (
        "Esta es una partición nueva. No representa la "
        "reconstrucción exacta de la partición histórica."
    ),
}

(DEST / "configuracion_particiones.json").write_text(
    json.dumps(resumen, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

## 11. Comprobaciones finales

In [ ]:
grupos_por_split = {
    split: set(df.loc[df["particion"] == split, "grupo_origen"])
    for split in ["train", "val", "test"]
}

assert grupos_por_split["train"].isdisjoint(grupos_por_split["val"])
assert grupos_por_split["train"].isdisjoint(grupos_por_split["test"])
assert grupos_por_split["val"].isdisjoint(grupos_por_split["test"])

for split in ["train", "val", "test"]:
    n_img = len(list((DEST / "images" / split).glob("*")))
    n_lab = len(list((DEST / "labels" / split).glob("*.txt")))
    assert n_img == n_lab, f"Descuadre de imágenes/etiquetas en {split}"

print("\n=== DATASET V3 CREADO ===")
print("Ubicación:", DEST)
print("Manifiesto:", ruta_manifest)
print("\nImágenes por partición:")
print(df.groupby("particion")["imagen"].count())
print("\nGrupos por partición:")
print(df.groupby("particion")["grupo_origen"].nunique())
print("\nArchivos guardados:")
for p in [
    DEST / "data.yaml",
    DEST / "manifest_particiones.csv",
    DEST / "configuracion_particiones.json",
]:
    print("-", p.name, "| existe:", p.exists())

display(df.head())

from pathlib import Path
from collections import Counter
import pandas as pd

DEST = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "dataset_oleocelosis_yolo_v3_reproducible"
)

EXTS = {".jpg", ".jpeg", ".png", ".webp"}
resumen = []
errores = []

for split in ["train", "val", "test"]:
    img_dir = DEST / "images" / split
    lab_dir = DEST / "labels" / split

    imagenes = sorted(
        p for p in img_dir.iterdir()
        if p.is_file() and p.suffix.lower() in EXTS
    )

    clases_por_imagen = Counter()
    cajas_por_clase = Counter()
    etiquetas_vacias = 0

    nombres_imagen = {p.stem for p in imagenes}
    nombres_etiqueta = {p.stem for p in lab_dir.glob("*.txt")}

    if nombres_imagen != nombres_etiqueta:
        errores.append(
            f"{split}: no coinciden los nombres de imágenes y etiquetas"
        )

    for img in imagenes:
        label = lab_dir / f"{img.stem}.txt"

        if not label.exists():
            continue

        lineas = label.read_text(encoding="utf-8").splitlines()

        if not lineas:
            etiquetas_vacias += 1
            errores.append(f"{split}: etiqueta vacía: {label.name}")
            continue

        clases_imagen = set()

        for num_linea, linea in enumerate(lineas, start=1):
            campos = linea.split()

            if len(campos) != 5:
                errores.append(
                    f"{split}/{label.name}, línea {num_linea}: "
                    "debe tener 5 valores"
                )
                continue

            try:
                clase = int(float(campos[0]))
                coords = [float(x) for x in campos[1:]]
            except ValueError:
                errores.append(
                    f"{split}/{label.name}, línea {num_linea}: "
                    "valores no numéricos"
                )
                continue

            if clase not in [0, 1]:
                errores.append(
                    f"{split}/{label.name}: clase inesperada {clase}"
                )

            if any(not (0 <= x <= 1) for x in coords):
                errores.append(
                    f"{split}/{label.name}: coordenadas fuera de [0,1]"
                )

            if coords[2] <= 0 or coords[3] <= 0:
                errores.append(
                    f"{split}/{label.name}: ancho o alto no positivo"
                )

            clases_imagen.add(clase)
            cajas_por_clase[clase] += 1

        for clase in clases_imagen:
            clases_por_imagen[clase] += 1

    resumen.append({
        "particion": split,
        "imagenes": len(imagenes),
        "etiquetas": len(list(lab_dir.glob("*.txt"))),
        "imagenes_con_clase_0": clases_por_imagen[0],
        "imagenes_con_clase_1": clases_por_imagen[1],
        "cajas_clase_0": cajas_por_clase[0],
        "cajas_clase_1": cajas_por_clase[1],
        "etiquetas_vacias": etiquetas_vacias,
    })

df_auditoria = pd.DataFrame(resumen)

print("=== RESUMEN DE AUDITORÍA ===")
display(df_auditoria)

print("\n=== RESULTADO ===")
if errores:
    print(f"Se encontraron {len(errores)} observaciones.")
    for e in errores[:30]:
        print("-", e)
    if len(errores) > 30:
        print("Se muestran solo las primeras 30.")
else:
    print("Todas las etiquetas superaron las comprobaciones básicas.")

# Guardar resultados de auditoría en Drive
ruta_auditoria = DEST / "auditoria_etiquetas.csv"
df_auditoria.to_csv(ruta_auditoria, index=False, encoding="utf-8-sig")
print("\nAuditoría guardada en:", ruta_auditoria)

from pathlib import Path
from PIL import Image, ImageDraw
import random
import matplotlib.pyplot as plt
import math

DEST = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "dataset_oleocelosis_yolo_v3_reproducible"
)

random.seed(42)
muestras = []

# Seleccionar imágenes de cada partición y clase
for split in ["train", "val", "test"]:
    img_dir = DEST / "images" / split
    lab_dir = DEST / "labels" / split

    por_clase = {0: [], 1: []}

    for img_path in img_dir.iterdir():
        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".webp"]:
            continue

        label_path = lab_dir / f"{img_path.stem}.txt"
        if not label_path.exists():
            continue

        clases = set()
        for linea in label_path.read_text().splitlines():
            campos = linea.split()
            if len(campos) == 5:
                clases.add(int(float(campos[0])))

        for clase in clases:
            por_clase[clase].append((img_path, label_path))

    # Hasta 2 ejemplos por clase y partición
    for clase in [0, 1]:
        candidatos = por_clase[clase]
        random.shuffle(candidatos)
        muestras.extend([
            (split, clase, img, lab)
            for img, lab in candidatos[:2]
        ])

# Dibujar las cajas de referencia
ncols = 3
nrows = math.ceil(len(muestras) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 5 * nrows))
axes = list(axes.flatten())

for ax, (split, clase_objetivo, img_path, label_path) in zip(axes, muestras):
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    W, H = img.size

    for linea in label_path.read_text().splitlines():
        campos = linea.split()
        if len(campos) != 5:
            continue

        clase = int(float(campos[0]))
        xc, yc, bw, bh = map(float, campos[1:])

        x1 = (xc - bw / 2) * W
        y1 = (yc - bh / 2) * H
        x2 = (xc + bw / 2) * W
        y2 = (yc + bh / 2) * H

        color = "lime" if clase == 0 else "red"
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)

    ax.imshow(img)
    ax.set_title(
        f"{split.upper()} | {img_path.name}\n"
        f"Clase seleccionada: {clase_objetivo} "
        f"({'sin' if clase_objetivo == 0 else 'con'} oleocelosis)",
        fontsize=9
    )
    ax.axis("off")

for ax in axes[len(muestras):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

from ultralytics import YOLO
from pathlib import Path
import shutil
import json
import time

# Dataset V3 auditado
DATA_YAML = (
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "dataset_oleocelosis_yolo_v3_reproducible/data.yaml"
)

# Carpeta persistente de resultados
BASE_DRIVE = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "resultados_yolo_v3_reproducible"
)
BASE_DRIVE.mkdir(parents=True, exist_ok=True)

# Verificar que existe la configuración
assert Path(DATA_YAML).exists(), f"No se encontró: {DATA_YAML}"

# Modelo base
model = YOLO("yolo11n.pt")

# Entrenamiento
inicio = time.time()

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    seed=42,
    deterministic=True,
    project="/content/smartfruit_runs",
    name="yolo_v3_reproducible",
    exist_ok=True,
    plots=True,
    verbose=True
)

duracion_min = round((time.time() - inicio) / 60, 2)

# Copiar el entrenamiento completo a Drive
RUN_DIR = Path("/content/smartfruit_runs/yolo_v3_reproducible")
DEST_RUN = BASE_DRIVE / "entrenamiento"

if DEST_RUN.exists():
    shutil.rmtree(DEST_RUN)

shutil.copytree(RUN_DIR, DEST_RUN)

# Guardar resumen de configuración
resumen = {
    "modelo": "YOLO11n",
    "dataset": "SmartFruit V3 reproducible",
    "epochs_maximas": 50,
    "imgsz": 640,
    "batch": 16,
    "patience": 10,
    "seed": 42,
    "duracion_minutos": duracion_min,
    "pesos_best": str(DEST_RUN / "weights" / "best.pt"),
    "pesos_last": str(DEST_RUN / "weights" / "last.pt")
}

with open(BASE_DRIVE / "configuracion_entrenamiento.json", "w") as f:
    json.dump(resumen, f, indent=4, ensure_ascii=False)

print("Entrenamiento terminado.")
print("Duración (minutos):", duracion_min)
print("Resultados guardados en:", DEST_RUN)
print("Mejores pesos:", DEST_RUN / "weights" / "best.pt")

from ultralytics import YOLO
from pathlib import Path
import json
import shutil

BASE_DRIVE = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "resultados_yolo_v3_reproducible"
)

DATA_YAML = (
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "dataset_oleocelosis_yolo_v3_reproducible/data.yaml"
)

WEIGHTS = BASE_DRIVE / "entrenamiento" / "weights" / "best.pt"
assert WEIGHTS.exists(), f"No se encontraron los pesos: {WEIGHTS}"
assert Path(DATA_YAML).exists(), "No se encontró data.yaml"

# Cargar el modelo entrenado
model = YOLO(str(WEIGHTS))

# Evaluación exclusiva del conjunto test
metrics = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    batch=16,
    plots=True,
    project=str(BASE_DRIVE),
    name="evaluacion_test",
    exist_ok=True,
    verbose=True
)

# Guardar las métricas principales en JSON
resumen = {
    "modelo": "YOLO11n",
    "dataset": "SmartFruit V3 reproducible",
    "conjunto_evaluado": "test",
    "pesos": str(WEIGHTS),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "metricas_por_clase": {}
}

# Métricas por clase, según los nombres registrados en el modelo
for i, nombre in model.names.items():
    resumen["metricas_por_clase"][nombre] = {
        "precision": float(metrics.box.p[i]),
        "recall": float(metrics.box.r[i]),
        "mAP50": float(metrics.box.ap50[i]),
        "mAP50_95": float(metrics.box.ap[i])
    }

ruta_json = BASE_DRIVE / "metricas_test_v3.json"
with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=4, ensure_ascii=False)

print("\n=== RESULTADOS TEST V3 ===")
print(json.dumps(resumen, indent=4, ensure_ascii=False))
print("\nMétricas guardadas en:", ruta_json)
print("Gráficos y resultados en:", BASE_DRIVE / "evaluacion_test")

from pathlib import Path
from PIL import Image

carpeta = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO/"
    "resultados_yolo_v3_reproducible/evaluacion_test"
)

print("Archivos disponibles:")
for f in carpeta.glob("*"):
    print(f.name)

from ultralytics import YOLO
from pathlib import Path
import numpy as np

base = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO"
)

modelo_path = (
    base / "resultados_yolo_v3_reproducible"
    / "entrenamiento" / "weights" / "best.pt"
)

data_yaml = (
    base / "dataset_oleocelosis_yolo_v3_reproducible"
    / "data.yaml"
)

modelo = YOLO(str(modelo_path))

metricas = modelo.val(
    data=str(data_yaml),
    split="test",
    imgsz=640,
    batch=16,
    plots=False,
    verbose=False
)

print("Clases:", modelo.names)
print("\nMatriz de confusión:")
print(np.array(metricas.confusion_matrix.matrix))

print("\nMétricas por clase:")
print(metricas.box.class_result(0))
print(metricas.box.class_result(1))

from ultralytics import YOLO
from pathlib import Path
import numpy as np

base = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO"
)

modelo = YOLO(str(
    base / "resultados_yolo_v3_reproducible"
    / "entrenamiento" / "weights" / "best.pt"
))

resultado = modelo.val(
    data=str(
        base / "dataset_oleocelosis_yolo_v3_reproducible"
        / "data.yaml"
    ),
    split="test",
    imgsz=640,
    batch=16,
    plots=True,
    verbose=False
)

cm = np.array(resultado.confusion_matrix.matrix)

print("Clases:", modelo.names)
print("\nMatriz de confusión:")
print(cm)
print("\nDirectorio de resultados:")
print(resultado.save_dir)

import numpy as np

cm = np.array(resultado.confusion_matrix.matrix)
nombres = ["sin oleocelosis", "con oleocelosis", "fondo"]

print("Filas = clase real | Columnas = clase predicha\n")
print(f"{'Real / Predicha':<22}", end="")
for nombre in nombres:
    print(f"{nombre:>22}", end="")
print()

for i, nombre in enumerate(nombres):
    print(f"{nombre:<22}", end="")
    for j in range(cm.shape[1]):
        print(f"{int(cm[i, j]):>22}", end="")
    print()

from pathlib import Path
from ultralytics import YOLO
import yaml

base = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO"
)

data_yaml = (
    base / "dataset_oleocelosis_yolo_v3_reproducible"
    / "data.yaml"
)

with open(data_yaml, "r") as f:
    config = yaml.safe_load(f)

test_dir = Path(config["path"]) / config["test"]
if not test_dir.is_absolute():
    test_dir = Path(config["path"]) / test_dir

imagenes = sorted(
    list(test_dir.glob("*.jpg")) +
    list(test_dir.glob("*.jpeg")) +
    list(test_dir.glob("*.png"))
)

modelo = YOLO(str(
    base / "resultados_yolo_v3_reproducible"
    / "entrenamiento" / "weights" / "best.pt"
))

print("Imágenes encontradas:", len(imagenes))

for img in imagenes[:10]:
    label = (
        Path(config["path"]) / "labels" / "test"
        / f"{img.stem}.txt"
    )

    reales = 0
    if label.exists():
        reales = sum(
            1 for linea in label.read_text().splitlines()
            if linea.strip()
        )

    pred = modelo.predict(
        source=str(img),
        imgsz=640,
        conf=0.25,
        verbose=False
    )[0]

    predichas = len(pred.boxes) if pred.boxes is not None else 0

    print(
        f"{img.name} | cajas reales: {reales} "
        f"| cajas predichas: {predichas}"
    )

from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO
import yaml
import numpy as np

base = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO"
)

dataset = base / "dataset_oleocelosis_yolo_v3_reproducible"
pesos = (
    base / "resultados_yolo_v3_reproducible"
    / "entrenamiento" / "weights" / "best.pt"
)

with open(dataset / "data.yaml", "r") as f:
    cfg = yaml.safe_load(f)

test_dir = Path(cfg["path"]) / cfg["test"]
if not test_dir.is_absolute():
    test_dir = dataset / cfg["test"]

labels_dir = dataset / "labels" / "test"
imagenes = sorted(
    p for p in test_dir.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
)

modelo = YOLO(str(pesos))
nombres = modelo.names

def iou(caja1, caja2):
    # Cajas en formato xyxy
    x1 = max(caja1[0], caja2[0])
    y1 = max(caja1[1], caja2[1])
    x2 = min(caja1[2], caja2[2])
    y2 = min(caja1[3], caja2[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = max(0, caja1[2]-caja1[0]) * max(0, caja1[3]-caja1[1])
    a2 = max(0, caja2[2]-caja2[0]) * max(0, caja2[3]-caja2[1])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0

conteo = defaultdict(lambda: {"reales": 0, "detectados": 0, "omitidos": 0})
iou_umbral = 0.5

for img in imagenes:
    label = labels_dir / f"{img.stem}.txt"
    reales = []

    if label.exists():
        for linea in label.read_text().splitlines():
            valores = linea.split()
            if len(valores) != 5:
                continue
            clase, xc, yc, w, h = map(float, valores)
            W, H = modelo.predictor.imgsz if False else (0, 0)
            reales.append((int(clase), xc, yc, w, h))

    pred = modelo.predict(
        source=str(img), imgsz=640, conf=0.001, iou=0.7,
        verbose=False
    )[0]

    if pred.boxes is None:
        pred_cajas = []
    else:
        pred_cajas = [
            (
                int(b.cls.item()),
                b.xyxy[0].cpu().numpy(),
                float(b.conf.item())
            )
            for b in pred.boxes
        ]

    # Convertir las anotaciones normalizadas a píxeles originales
    from PIL import Image
    with Image.open(img) as im:
        W, H = im.size

    reales_xyxy = []
    for clase, xc, yc, w, h in reales:
        reales_xyxy.append((
            clase,
            np.array([
                (xc-w/2)*W, (yc-h/2)*H,
                (xc+w/2)*W, (yc+h/2)*H
            ])
        ))

    usadas = set()

    for clase, caja_real in reales_xyxy:
        conteo[clase]["reales"] += 1
        mejor_iou = 0
        mejor_idx = None

        for idx, (clase_pred, caja_pred, confianza) in enumerate(pred_cajas):
            if idx in usadas or clase_pred != clase:
                continue
            valor_iou = iou(caja_real, caja_pred)
            if valor_iou > mejor_iou:
                mejor_iou, mejor_idx = valor_iou, idx

        if mejor_idx is not None and mejor_iou >= iou_umbral:
            usadas.add(mejor_idx)
            conteo[clase]["detectados"] += 1
        else:
            conteo[clase]["omitidos"] += 1

print("Coincidencias por clase con IoU >= 0.5")
for clase, nombre in nombres.items():
    d = conteo[clase]
    print(
        f"{nombre}: cajas reales={d['reales']}, "
        f"coincidencias={d['detectados']}, "
        f"no detectadas={d['omitidos']}"
    )

import numpy as np
from pathlib import Path
from ultralytics import YOLO

base = Path(
    "/content/drive/MyDrive/MAESTRÍA UTEC/CICLO V/"
    "PROYECTO INTEGRADOR/SMARTFRUIT MODELO"
)

modelo = YOLO(str(
    base / "resultados_yolo_v3_reproducible"
    / "entrenamiento" / "weights" / "best.pt"
))

resultado = modelo.val(
    data=str(
        base / "dataset_oleocelosis_yolo_v3_reproducible"
        / "data.yaml"
    ),
    split="test",
    imgsz=640,
    batch=16,
    conf=0.001,
    iou=0.7,
    plots=False,
    save_json=True,
    verbose=False
)

print("\nMétricas oficiales:")
print("Precision:", resultado.box.mp)
print("Recall:", resultado.box.mr)
print("mAP@50:", resultado.box.map50)
print("mAP@50-95:", resultado.box.map)

print("\nMétricas por clase:")
for i, nombre in modelo.names.items():
    print(nombre, resultado.box.class_result(i))

print("\nDirectorio de evaluación:", resultado.save_dir)